# One subject: channels, segmentation, Dice

This notebook walks through **UCSF-PDGM subject 0005**, one of the three subjects used by `make smoke-segment`.

It loads the four bias-corrected MRI channels, runs the same MONAI BraTS segmenter as the smoke test, and compares the predicted enhancing tumor with the dataset reference.

Cohort figures and tables stay in the Make targets listed at the end. This notebook does not run FreeSurfer, the n=50 cohort, or TumorSynth.


## Before you run it

From the repository root, with [UCSF-PDGM v5](../data/README.md) downloaded:

```bash
uv run python scripts/extract_ucsfpdgm.py \
    --src /path/to/UCSF-PDGM-v5 \
    --dst ./data/ucsf_pdgm_cohort50
```

`--src` is the directory that contains `UCSF-PDGM-XXXX_nifti/` folders. The cell below expects `data/ucsf_pdgm_cohort50/sub-0005/`.

The first segmentation call downloads the MONAI `brats_mri_segmentation` bundle into `~/.cache/hpgs/monai_bundles` and then runs on MPS, CUDA, or CPU. On this project's Apple Silicon machines that inference takes a few minutes.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from hpgs.io import load_nifti, resolve_subject_inputs
from hpgs.metrics import dice
from hpgs.segment import derive_subregions
from hpgs.segment.unified import predict_brats3

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

SUBJECT_ID = "0005"
COHORT_ROOT = REPO_ROOT / "data" / "ucsf_pdgm_cohort50"
CHANNELS = ("T1_bias", "T1c_bias", "T2_bias", "FLAIR_bias")

subject_dir = COHORT_ROOT / f"sub-{SUBJECT_ID}"
if not subject_dir.is_dir():
    raise FileNotFoundError(
        f"{subject_dir} is not present. Extract UCSF-PDGM v5 first:\n"
        "uv run python scripts/extract_ucsfpdgm.py "
        "--src /path/to/UCSF-PDGM-v5 --dst ./data/ucsf_pdgm_cohort50"
    )

inputs = resolve_subject_inputs(
    COHORT_ROOT,
    SUBJECT_ID,
    channel_names=list(CHANNELS),
    tumor_segmentation_name="tumor_segmentation",
)
print(f"sub-{SUBJECT_ID} channels:")
for name, path in inputs.channels.items():
    print(f"  {name}: {path.relative_to(REPO_ROOT)}")


## Four channels, one axial slice

The extracted UCSF-PDGM volumes are already skull-stripped and on a 1 mm grid. The slice is the axial plane that contains the most enhancing-tumor voxels in the dataset reference, so the lesion is in view. Display is radiological: the patient's right is on the left of the image.


In [ ]:
volumes = {name: load_nifti(path)[0] for name, path in inputs.channels.items()}
reference, _affine = load_nifti(inputs.tumor_segmentation)
reference_masks = derive_subregions(reference)


def axial_index(volume: np.ndarray, mask: np.ndarray) -> int:
    """Index of the axial plane with the most mask voxels."""
    axis = int(np.argmin(volume.shape[:3]))
    other = tuple(i for i in range(3) if i != axis)
    counts = np.asarray(mask).sum(axis=other)
    if int(counts.max()) == 0:
        return volume.shape[axis] // 2
    return int(np.argmax(counts))


def axial_slice(volume: np.ndarray, index: int) -> np.ndarray:
    axis = int(np.argmin(volume.shape[:3]))
    slc = np.take(volume, index, axis=axis)
    return np.fliplr(slc)


slice_index = axial_index(volumes["T1c_bias"], reference_masks.et)

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4), constrained_layout=True)
for ax, name in zip(axes, CHANNELS, strict=True):
    ax.imshow(axial_slice(volumes[name], slice_index), cmap="gray")
    ax.set_title(name.replace("_bias", ""))
    ax.axis("off")
fig.suptitle(f"sub-{SUBJECT_ID}, axial slice {slice_index}")
plt.show()


## The same segmenter as `make smoke-segment`

`predict_brats3` is the MONAI Bundle BraTS SegResNet used for the article. `device="auto"` selects CUDA, then MPS, then CPU. Channel order is handled inside that function: the bundle expects T1c before T1.


In [ ]:
result = predict_brats3(
    t1=inputs.channels["T1_bias"],
    t1c=inputs.channels["T1c_bias"],
    t2=inputs.channels["T2_bias"],
    flair=inputs.channels["FLAIR_bias"],
    backend="monai_bundle",
    device="auto",
)
predicted = derive_subregions(result.label_map)
reference_masks = derive_subregions(reference)

print(f"backend: {result.backend} {result.bundle_version}")
print(f"{'compartment':<14}{'Dice':>8}")
for name, pred_mask, ref_mask in (
    ("whole tumor", predicted.wt, reference_masks.wt),
    ("tumor core", predicted.tc, reference_masks.tc),
    ("enhancing tumor", predicted.et, reference_masks.et),
):
    print(f"{name:<14}{dice(pred_mask, ref_mask):>8.3f}")


## Enhancing tumor on post-contrast T1

The outline is the dataset reference. The filled region is the predicted enhancing tumor. Dice for that compartment is the number printed above as `enhancing tumor`.


In [ ]:
t1c = axial_slice(volumes["T1c_bias"], slice_index)
pred_et = axial_slice(predicted.et, slice_index)
ref_et = axial_slice(reference_masks.et, slice_index)

fig, ax = plt.subplots(figsize=(5.2, 5.2), constrained_layout=True)
ax.imshow(t1c, cmap="gray")
ax.imshow(np.ma.masked_where(pred_et == 0, pred_et), cmap="autumn", alpha=0.55)
ax.contour(ref_et, levels=[0.5], colors="cyan", linewidths=1.2)
ax.set_title(f"sub-{SUBJECT_ID} enhancing tumor\nfilled = prediction, outline = reference")
ax.axis("off")
plt.show()


## Where the article figures come from

This notebook stops at one subject. To write the same segmentation the smoke test saves:

```bash
make smoke-segment SMOKE_SUBJECTS=0005 SMOKE_DEVICE=mps
```

The cohort run, FreeSurfer parcellation, and the computed manuscript figures are separate Make targets. [`paper/figs/README.md`](../paper/figs/README.md) maps each figure and table to its producer. [`docs/software_installation.md`](../docs/software_installation.md) is the runbook. `make paper` rebuilds the author PDF from the figures already in `paper/figs/`.
